# as-strided-windowing composite — cx9: as_strided windowing with stride > 1

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `as-strided-windowing`, `conv-stride-downsample`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "as-strided-windowing"
DD_ATOM_IDS = ["as-strided-windowing", "conv-stride-downsample"]
DD_SUBTOPICS = ["PyTorch: as_strided windowing", "CNN: Stride downsample arithmetic"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

Stride-1 windowing puts the source's element-stride `s_w` on BOTH the new `OW` axis and the `K` axis. For stride-`S` windowing, the change is surgical: **multiply the `OW` stride by `S`**, leave the `K` stride alone.

```
# stride-1 windowing (cx8):
stride=(s_b, s_ic, s_w,     s_w)
# stride-S windowing (this drill):
stride=(s_b, s_ic, s_w * S, s_w)
```

The new `OW` stride means 'advance by `S` elements of the original W axis when moving to the next window' — i.e. SKIP `S - 1` positions between window starts. The `K` stride stays the same: within a window you still walk one element at a time.

**Output length** (`conv-stride-downsample` atom): `OW = (W - K) // S + 1`. Floor division drops any partial trailing window.

**Why the multiply, not a separate skip step.** as_strided does the skipping FOR you via the stride argument. You never write a loop, never call `step`. The view directly indexes into the right memory cells.

**Stride-1 special case.** When `S = 1`, `s_w * 1 == s_w` and the formula collapses to the stride-1 case — same atom, same code path.

### Composite Exercise — as_strided windowing with stride > 1

**Atoms exercised together**: `as-strided-windowing`, `conv-stride-downsample`

Implement `cx9_strided_windows(x, K, S)`.

- `x`: float tensor of shape `(B, IC, W)`.
- `K`: kernel width.
- `S`: stride (>= 1).

Return a `(B, IC, OW, K)` view where `OW = (W - K) // S + 1`, each window starts `S` elements after the previous one, and the view shares storage with `x` (no copy).

1. **Output length** — apply the strided-conv formula `OW = (W - K) // S + 1`.
2. **Read source strides** — `s_b, s_ic, s_w = x.stride()`.
3. **Strided as_strided** — call `x.as_strided(size=(B, IC, OW, K), stride=(s_b, s_ic, s_w * S, s_w))`. The `s_w * S` is the load-bearing piece — that's where the skipping happens.

The test:
- Verifies `OW` matches the formula across many `(W, K, S)` configs.
- Checks `win[..., k]` equals `x[..., k*S : k*S + K]` for several windows.
- Confirms `data_ptr` matches (no copy).
- Cross-checks `einsum(windows, weight)` against `F.conv1d(x, weight, stride=S)`.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx9_strided_windows(x, K, S):
    raise NotImplementedError

def _test_cx9():
    from torch.nn import functional as F

    # Case A: hand-built, S=2, K=3. OW = (8-3)//2 + 1 = 3.
    x = t.arange(1.0, 9.0).reshape(1, 1, 8).contiguous()  # [1..8]
    win = cx9_strided_windows(x, K=3, S=2)
    assert tuple(win.shape) == (1, 1, 3, 3), f'shape: {tuple(win.shape)}'
    # Windows: [1,2,3], [3,4,5], [5,6,7].
    assert t.allclose(win[0, 0, 0], t.tensor([1.0, 2.0, 3.0]))
    assert t.allclose(win[0, 0, 1], t.tensor([3.0, 4.0, 5.0]))
    assert t.allclose(win[0, 0, 2], t.tensor([5.0, 6.0, 7.0]))

    # Case B: no-copy (must share storage with x).
    assert win.data_ptr() == x.data_ptr(), 'windows must be a view (share storage with x)'

    # Case C: stride == kernel size — non-overlapping tiles.
    x = t.arange(1.0, 13.0).reshape(1, 1, 12).contiguous()
    win = cx9_strided_windows(x, K=4, S=4)
    assert tuple(win.shape) == (1, 1, 3, 4)  # (12-4)//4 + 1 = 3.
    assert t.allclose(win[0, 0, 0], t.tensor([1.0, 2.0, 3.0, 4.0]))
    assert t.allclose(win[0, 0, 1], t.tensor([5.0, 6.0, 7.0, 8.0]))
    assert t.allclose(win[0, 0, 2], t.tensor([9.0, 10.0, 11.0, 12.0]))

    # Case D: stride-1 special case — equals plain stride-1 windowing.
    x = t.arange(1.0, 11.0).reshape(1, 1, 10).contiguous()
    win = cx9_strided_windows(x, K=3, S=1)
    assert tuple(win.shape) == (1, 1, 8, 3)  # (10-3)//1 + 1 = 8.
    for k in range(8):
        assert t.allclose(win[0, 0, k], x[0, 0, k:k+3])

    # Case E: off-by-one cross-check vs F.conv1d for many configs.
    rng = t.Generator().manual_seed(9)
    for B, IC, W, K, S, OC in [
        (1, 1, 10, 3, 2, 1),
        (2, 3, 20, 5, 3, 4),
        (3, 2, 32, 3, 2, 2),
        (1, 4,  9, 3, 3, 2),  # stride == kernel
        (1, 1, 11, 4, 2, 1),  # tests trailing-window drop
    ]:
        x = t.randn(B, IC, W, generator=rng)
        win = cx9_strided_windows(x, K, S)
        OW = (W - K) // S + 1
        assert tuple(win.shape) == (B, IC, OW, K), (
            f'OW formula wrong: predicted {OW}, got {win.shape[-2]} for W={W} K={K} S={S}'
        )
        # Spot-check a window.
        for k in range(OW):
            assert t.allclose(win[..., k, :], x[..., k*S : k*S + K]), (
                f'window {k} content mismatch — did you forget s_w * S on the OW axis?'
            )
        # Cross-check vs F.conv1d.
        weight = t.randn(OC, IC, K, generator=rng)
        y_manual = einops.einsum(win, weight, 'b ic ow kw, oc ic kw -> b oc ow')
        y_native = F.conv1d(x, weight, stride=S)
        assert t.allclose(y_manual, y_native, atol=1e-4)

    # Case F: off-by-one trap — for W=32 K=3 S=2, OW must be 15, NOT 16.
    x = t.zeros(1, 1, 32)
    win = cx9_strided_windows(x, K=3, S=2)
    assert win.shape[-2] == 15, f'expected 15 (not 16), got {win.shape[-2]}'
    _dd_passed.add('cx9')

_test_cx9()

<details><summary>Show solution — cx9</summary>

```python
def cx9_strided_windows(x, K, S):
    B, IC, W = x.shape
    # Atom B (conv-stride-downsample): floor division + 1 for the leading window.
    OW = (W - K) // S + 1
    s_b, s_ic, s_w = x.stride()
    # Atom A (as-strided-windowing): the OW-axis stride is s_w * S — the multiply IS the
    # skipping. The K-axis stride stays s_w (walk within a window).
    return x.as_strided(
        size=(B, IC, OW, K),
        stride=(s_b, s_ic, s_w * S, s_w),
    )
```

Two-line solution; two atoms. The strided-conv formula gives `OW`; the as_strided call uses `s_w * S` on the new `OW` axis to skip `S - 1` elements between window starts. The `K` axis stride is still `s_w` — within a window you advance one element at a time. Hard-coding `S` on both axes is a common bug (would skip *within* the window too).
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx9'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx9',
        'subtopics': ["PyTorch: as_strided windowing", "CNN: Stride downsample arithmetic"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()